# YOLOv8 GPU Inference Stack — Colab Live Benchmark

**Runtime required**: Google Colab T4 or A100  
**Purpose**: Run the full inference stack on a real GPU and measure production-grade metrics

## What this notebook does
1. Verifies GPU + installs dependencies
2. Loads the trained YOLOv8-Large wafer defect model from Google Drive
3. Benchmarks **direct GPU inference** (FP16) — latency at P50/P95/P99, throughput (FPS)
4. Starts a **FastAPI server** on port 9000 (background subprocess)
5. Runs an **async HTTP load test** — concurrency sweep 1→2→4→8 threads
6. Monitors **GPU utilization and memory** under sustained load
7. Generates a publication-quality 6-panel results chart
8. Downloads all results as a zip

## Expected Results (T4 GPU, YOLOv8-Large 43.7M params)
| Mode | Latency P50 | P99 | FPS |
|------|------------|-----|-----|
| PyTorch FP16 (`half=True`) | ~20-30ms | ~35ms | ~35-50 |
| PyTorch FP32 (default) | ~45-55ms | ~60ms | ~18-22 |

> FP16 is enabled by default in this notebook. For ~8ms you need TensorRT FP16 export (separate notebook).  
> Run cells 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 in order.


In [ ]:
# ─── Cell 1: Verify GPU + Install dependencies ───────────────────────────────
import subprocess

# Show GPU info
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Install required packages
!pip install -q ultralytics>=8.1 fastapi uvicorn[standard] httpx pillow pynvml nest_asyncio

In [ ]:
# ─── Cell 2: Mount Google Drive + load model ─────────────────────────────────
# Your best.pt is already at: My Drive > YOLO > best.pt
# No uploads needed — just run this cell and authorize Drive access

from google.colab import drive
import shutil, os
from pathlib import Path

drive.mount('/content/drive', force_remount=False)

# Source: your Google Drive YOLO folder
DRIVE_MODEL = '/content/drive/MyDrive/YOLO/best.pt'
LOCAL_MODEL  = '/content/models/best.pt'

os.makedirs('/content/models', exist_ok=True)

if not os.path.exists(DRIVE_MODEL):
    raise FileNotFoundError(
        f'Model not found at {DRIVE_MODEL}\n'
        'Make sure best.pt is inside My Drive > YOLO > best.pt'
    )

# Copy to local content dir (faster inference than reading from Drive)
shutil.copy2(DRIVE_MODEL, LOCAL_MODEL)
size_mb = Path(LOCAL_MODEL).stat().st_size / 1e6

MODEL_PATH = LOCAL_MODEL
MODEL_TYPE  = 'pytorch'

print(f'Model loaded from Drive: {DRIVE_MODEL}')
print(f'Local copy:              {LOCAL_MODEL}  ({size_mb:.1f} MB)')
print(f'Model type:              PYTORCH (always — TensorRT engines are version-locked to the build environment)')


In [ ]:
# ─── Cell 3: Direct GPU inference benchmark (FP16) ───────────────────────────
# Measures model inference speed with FP16 (half precision) for T4 tensor cores
import time
import numpy as np
from ultralytics import YOLO
import torch

# Determine device
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
USE_HALF = torch.cuda.is_available()  # FP16 on GPU, FP32 on CPU

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
precision = 'FP16' if USE_HALF else 'FP32'
print(f'Device:    cuda:0 — {gpu_name}' if DEVICE == 0 else 'Device:    CPU')
print(f'Precision: {precision}')

print(f'Loading model: {MODEL_PATH}')
model = YOLO(MODEL_PATH)

# Generate synthetic 640x640 wafer image (3-channel uint8)
dummy_img = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)

N_WARMUP = 50
N_RUNS   = 300

print(f'Warming up ({N_WARMUP} runs on {gpu_name}, {precision})...')
for _ in range(N_WARMUP):
    model.predict(dummy_img, device=DEVICE, half=USE_HALF, verbose=False)

# Sync GPU before benchmarking
if torch.cuda.is_available():
    torch.cuda.synchronize()

print(f'Benchmarking ({N_RUNS} runs)...')
latencies_ms = []
for _ in range(N_RUNS):
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    model.predict(dummy_img, device=DEVICE, half=USE_HALF, verbose=False)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    latencies_ms.append((time.perf_counter() - t0) * 1000)

arr = np.array(latencies_ms)
direct_results = {
    'model_type': f'{MODEL_TYPE}_{precision}',
    'precision':  precision,
    'mean_ms':    round(float(arr.mean()), 3),
    'median_ms':  round(float(np.median(arr)), 3),
    'p95_ms':     round(float(np.percentile(arr, 95)), 3),
    'p99_ms':     round(float(np.percentile(arr, 99)), 3),
    'min_ms':     round(float(arr.min()), 3),
    'max_ms':     round(float(arr.max()), 3),
    'fps':        round(float(1000 / arr.mean()), 1),
    'std_ms':     round(float(arr.std()), 3),
}

print()
print(f'=== Direct GPU Inference Benchmark ({precision}) ===')
print(f'  GPU:        {gpu_name}')
print(f'  Model:      YOLOv8-Large (.pt) {precision}')
print(f'  Mean:       {direct_results["mean_ms"]:.2f} ms')
print(f'  Median P50: {direct_results["median_ms"]:.2f} ms')
print(f'  P95:        {direct_results["p95_ms"]:.2f} ms')
print(f'  P99:        {direct_results["p99_ms"]:.2f} ms')
print(f'  Throughput: {direct_results["fps"]:.0f} FPS')
print(f'\n  (FP32 would be ~50-55ms. TensorRT FP16 would be ~8ms.)')


In [ ]:
# ─── Cell 4: Write FastAPI server to file ────────────────────────────────────
# Server uses device=0 (GPU) and half=True (FP16) — matches Cell 3 settings

import torch as _torch
_use_half = _torch.cuda.is_available()

SERVER_CODE = f'''
import io, os, time, logging
import numpy as np
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import JSONResponse
from PIL import Image
from ultralytics import YOLO
import torch

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("wafer_api")

app = FastAPI(title="Wafer Defect Detection API", version="2.0")

MODEL_PATH = os.environ.get("MODEL_PATH", "models/best.pt")
DEVICE = 0 if torch.cuda.is_available() else "cpu"
USE_HALF = {_use_half}

_model = None
_inference_count = 0
_total_latency_ms = 0.0

@app.on_event("startup")
async def startup():
    global _model
    logger.info("Loading model: %s on device=%s half=%s", MODEL_PATH, DEVICE, USE_HALF)
    _model = YOLO(MODEL_PATH)
    # Warm up — first call is slow due to CUDA kernel compilation
    dummy = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)
    for _ in range(5):
        _model.predict(dummy, device=DEVICE, half=USE_HALF, verbose=False)
    logger.info("Model loaded and warmed up")

@app.get("/health")
def health():
    return {{"status": "healthy", "model_loaded": _model is not None, "inference_count": _inference_count}}

@app.get("/metrics")
def metrics():
    avg = round(_total_latency_ms / max(_inference_count, 1), 2)
    return {{"inference_count": _inference_count, "avg_inference_ms": avg}}

@app.post("/detect")
async def detect(file: UploadFile = File(...), confidence: float = 0.25):
    global _inference_count, _total_latency_ms
    data = await file.read()
    img = Image.open(io.BytesIO(data)).convert("RGB")
    t0 = time.perf_counter()
    results = _model.predict(np.array(img), device=DEVICE, half=USE_HALF, conf=confidence, verbose=False)
    latency_ms = (time.perf_counter() - t0) * 1000
    _inference_count += 1
    _total_latency_ms += latency_ms
    boxes = results[0].boxes
    detections = []
    if boxes is not None:
        for box in boxes:
            detections.append({{
                "class": int(box.cls[0]),
                "confidence": round(float(box.conf[0]), 4),
                "bbox": [round(float(x), 2) for x in box.xyxy[0].tolist()]
            }})
    return JSONResponse({{"detections": detections, "count": len(detections), "latency_ms": round(latency_ms, 2)}})
'''

with open('colab_server.py', 'w') as f:
    f.write(SERVER_CODE)

print(f'Server file written: colab_server.py (device=GPU, half={_use_half})')


In [ ]:
# ─── Cell 5: Start FastAPI server as subprocess ──────────────────────────────
import subprocess, time, httpx, os

API_PORT = 9000

# Initialize server_proc on first run
if 'server_proc' not in globals():
    server_proc = None

# Kill previous server_proc if this cell is being re-run
if server_proc is not None and server_proc.poll() is None:
    server_proc.terminate()
    try:
        server_proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        server_proc.kill()
        server_proc.wait()
    print('Previous server process stopped.')
    time.sleep(2)

env = os.environ.copy()
env['MODEL_PATH'] = MODEL_PATH

print(f'Starting FastAPI server on port {API_PORT}...')
print('(Server will warm up the model on GPU — may take 30-60s)')
server_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'colab_server:app',
     '--host', '0.0.0.0', '--port', str(API_PORT),
     '--timeout-keep-alive', '30',
     '--backlog', '64'],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Wait for server ready — max 180 seconds (model load + warmup)
BASE_URL = f'http://localhost:{API_PORT}'
for attempt in range(90):
    time.sleep(2)
    try:
        resp = httpx.get(f'{BASE_URL}/health', timeout=5)
        if resp.status_code == 200 and resp.json().get('model_loaded'):
            print(f'✓ Server ready after {(attempt+1)*2}s — {resp.json()}')
            break
    except Exception:
        if attempt % 5 == 0:
            print(f'  Waiting... ({(attempt+1)*2}s elapsed)')
        # Check if server process died
        if server_proc.poll() is not None:
            logs = server_proc.stdout.read(8192).decode(errors='replace')
            print('=== Server crashed! Logs: ===')
            print(logs)
            raise RuntimeError('Server process exited unexpectedly. Check logs above.')
else:
    try:
        server_proc.terminate()
        server_proc.wait(timeout=5)
        logs = server_proc.stdout.read(8192).decode(errors='replace')
    except Exception:
        logs = '(could not read logs)'
    print('=== Server logs ===')
    print(logs)
    raise RuntimeError('Server did not become healthy in 180s.')

print(f'FastAPI server is LIVE at {BASE_URL}')


In [ ]:
# ─── Cell 6: Async HTTP load test — concurrency sweep ────────────────────────
# Tests: 1, 2, 4, 8 concurrent clients (no 16/32 — single-worker server can't handle)
# Each concurrency level: 100 requests
# Measures: latency P50/P95/P99, throughput (req/sec), error rate

import asyncio, io, time
import numpy as np
import httpx
from PIL import Image
import nest_asyncio
nest_asyncio.apply()

def make_dummy_image_bytes(width=640, height=640):
    """Generate a synthetic wafer image as JPEG bytes."""
    arr = np.random.randint(100, 200, (height, width, 3), dtype=np.uint8)
    for _ in range(np.random.randint(0, 5)):
        cx, cy = np.random.randint(50, width-50), np.random.randint(50, height-50)
        r = np.random.randint(10, 40)
        y_grid, x_grid = np.ogrid[:height, :width]
        mask = (x_grid - cx)**2 + (y_grid - cy)**2 <= r**2
        arr[mask] = np.random.randint(20, 80)
    img = Image.fromarray(arr)
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=85)
    return buf.getvalue()

async def run_single_request(client: httpx.AsyncClient, image_bytes: bytes):
    t0 = time.perf_counter()
    try:
        resp = await client.post(
            f'{BASE_URL}/detect',
            files={'file': ('image.jpg', image_bytes, 'image/jpeg')},
            params={'confidence': 0.25},
            timeout=60.0   # 60s timeout — generous to avoid false errors
        )
        latency = (time.perf_counter() - t0) * 1000
        return {'latency_ms': latency, 'status': resp.status_code, 'error': None}
    except Exception as e:
        latency = (time.perf_counter() - t0) * 1000
        return {'latency_ms': latency, 'status': 0, 'error': str(e)}

async def run_concurrency_level(concurrency: int, n_requests: int = 100):
    """Run n_requests with `concurrency` parallel clients."""
    image_bytes_list = [make_dummy_image_bytes() for _ in range(min(concurrency, 8))]

    results = []
    t_start = time.perf_counter()

    async with httpx.AsyncClient() as client:
        for batch_start in range(0, n_requests, concurrency):
            batch_size = min(concurrency, n_requests - batch_start)
            tasks = []
            for i in range(batch_size):
                img = image_bytes_list[i % len(image_bytes_list)]
                tasks.append(run_single_request(client, img))
            batch_results = await asyncio.gather(*tasks)
            results.extend(batch_results)

    t_total = time.perf_counter() - t_start

    latencies = [r['latency_ms'] for r in results if r['error'] is None]
    errors = sum(1 for r in results if r['error'] is not None)
    arr = np.array(latencies) if latencies else np.array([0.0])

    return {
        'concurrency': concurrency,
        'n_requests': n_requests,
        'successful': len(latencies),
        'errors': errors,
        'error_rate_pct': round(errors / n_requests * 100, 1),
        'throughput_rps': round(len(latencies) / t_total, 1) if latencies else 0.0,
        'p50_ms': round(float(np.percentile(arr, 50)), 2) if latencies else 0.0,
        'p95_ms': round(float(np.percentile(arr, 95)), 2) if latencies else 0.0,
        'p99_ms': round(float(np.percentile(arr, 99)), 2) if latencies else 0.0,
        'mean_ms': round(float(arr.mean()), 2) if latencies else 0.0,
    }

# Only test concurrency levels that a single-worker server can handle
# At ~20-30ms per request, max throughput is ~35-50 rps
# c=8 means 8 requests queue up → ~240ms max wait → totally fine
CONCURRENCY_LEVELS = [1, 2, 4, 8]
load_test_results = []

# Verify server is still alive
try:
    resp = httpx.get(f'{BASE_URL}/health', timeout=5)
    assert resp.status_code == 200
    print(f'Server confirmed healthy: {resp.json()}')
except Exception as e:
    raise RuntimeError(f'Server is not responding! Re-run Cell 5 first. Error: {e}')

print()
print('=== Async HTTP Load Test ===')
print(f'Endpoint: {BASE_URL}/detect')
print(f'Concurrency sweep: {CONCURRENCY_LEVELS}')
print(f'100 requests per level, 60s timeout per request')
print()

for c in CONCURRENCY_LEVELS:
    print(f'  Running concurrency={c}...', end='', flush=True)
    res = asyncio.get_event_loop().run_until_complete(run_concurrency_level(c, n_requests=100))
    load_test_results.append(res)
    print(f'  P50={res["p50_ms"]}ms  P99={res["p99_ms"]}ms  {res["throughput_rps"]}rps  errors={res["errors"]}')

print('\nLoad test complete.')


In [ ]:
# ─── Cell 7: GPU monitoring — 30s sustained load ─────────────────────────────
import threading, pynvml, httpx, io, time
import numpy as np

# Verify server health first
try:
    resp = httpx.get(f'{BASE_URL}/health', timeout=5)
    assert resp.status_code == 200
    print(f'Server healthy: {resp.json()}')
except Exception as e:
    raise RuntimeError(f'Server not responding — re-run Cell 5. Error: {e}')

pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0)

gpu_samples = []
stop_event = threading.Event()

def gpu_monitor():
    while not stop_event.is_set():
        try:
            util = pynvml.nvmlDeviceGetUtilizationRates(handle)
            mem = pynvml.nvmlDeviceGetMemoryInfo(handle)
            temp = pynvml.nvmlDeviceGetTemperature(handle, pynvml.NVML_TEMPERATURE_GPU)
            gpu_samples.append({
                'ts': time.time(),
                'gpu_util_pct': util.gpu,
                'mem_used_gb': round(mem.used / 1e9, 2),
                'mem_total_gb': round(mem.total / 1e9, 2),
                'temp_c': temp,
            })
        except Exception:
            pass
        time.sleep(1)

monitor_thread = threading.Thread(target=gpu_monitor, daemon=True)
monitor_thread.start()

print('Running 30-second sustained load (concurrency=4) while sampling GPU...')
LOAD_DURATION_SEC = 30
CONCURRENCY = 4  # safe level — no server overload

async def sustained_load():
    t_end = time.time() + LOAD_DURATION_SEC
    request_count = 0
    error_count = 0
    img_bytes = make_dummy_image_bytes()
    async with httpx.AsyncClient() as client:
        while time.time() < t_end:
            tasks = [run_single_request(client, img_bytes) for _ in range(CONCURRENCY)]
            results = await asyncio.gather(*tasks)
            request_count += CONCURRENCY
            error_count += sum(1 for r in results if r['error'] is not None)
    return request_count, error_count

total_requests, total_errors = asyncio.get_event_loop().run_until_complete(sustained_load())
stop_event.set()
monitor_thread.join(timeout=3)

if gpu_samples:
    util_vals = [s['gpu_util_pct'] for s in gpu_samples]
    mem_vals = [s['mem_used_gb'] for s in gpu_samples]
    temp_vals = [s['temp_c'] for s in gpu_samples]
    print(f'\n=== GPU Monitoring (30s sustained load) ===')
    print(f'  Total requests sent:      {total_requests}')
    print(f'  Errors:                   {total_errors}')
    print(f'  GPU Utilization: avg={np.mean(util_vals):.0f}%  peak={max(util_vals)}%')
    print(f'  GPU Memory:      avg={np.mean(mem_vals):.2f}  peak={max(mem_vals):.2f} GB')
    print(f'  GPU Temp:        avg={np.mean(temp_vals):.0f}°C  peak={max(temp_vals)}°C')
    print(f'  Samples collected: {len(gpu_samples)}')
else:
    print('WARNING: No GPU samples collected')


In [ ]:
# ─── Cell 8: Publication-quality results chart ───────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

concurrencies = [r['concurrency'] for r in load_test_results]
p50s  = [r['p50_ms'] for r in load_test_results]
p95s  = [r['p95_ms'] for r in load_test_results]
p99s  = [r['p99_ms'] for r in load_test_results]
rps   = [r['throughput_rps'] for r in load_test_results]

gpu_label = ''
import torch
if torch.cuda.is_available():
    gpu_label = torch.cuda.get_device_name(0)

precision = direct_results.get('precision', 'FP16')

fig = plt.figure(figsize=(18, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# Panel 1: Latency vs Concurrency
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(concurrencies, p50s, 'o-', color='#22c55e', linewidth=2, markersize=6, label='P50')
ax1.plot(concurrencies, p95s, 's--', color='#f59e0b', linewidth=2, markersize=6, label='P95')
ax1.plot(concurrencies, p99s, '^:', color='#ef4444', linewidth=2, markersize=6, label='P99')
ax1.set_xlabel('Concurrent Clients')
ax1.set_ylabel('Latency (ms)')
ax1.set_title('API Latency vs Concurrency', fontweight='bold')
ax1.legend()
ax1.set_xticks(concurrencies)
ax1.grid(True, alpha=0.3)

# Panel 2: Throughput vs Concurrency
ax2 = fig.add_subplot(gs[0, 1])
bars = ax2.bar([str(c) for c in concurrencies], rps, color='#3b82f6', edgecolor='white')
ax2.set_xlabel('Concurrent Clients')
ax2.set_ylabel('Requests per Second')
ax2.set_title('Throughput vs Concurrency', fontweight='bold')
for bar, val in zip(bars, rps):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.0f}', ha='center', fontsize=9, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# Panel 3: Direct inference benchmark
ax3 = fig.add_subplot(gs[0, 2])
metrics_direct = ['Mean', 'Median', 'P95', 'P99']
values_direct = [direct_results['mean_ms'], direct_results['median_ms'],
                 direct_results['p95_ms'], direct_results['p99_ms']]
colors_direct = ['#3b82f6', '#22c55e', '#f59e0b', '#ef4444']
bars3 = ax3.bar(metrics_direct, values_direct, color=colors_direct, edgecolor='white')
ax3.set_ylabel('Latency (ms)')
ax3.set_title(f'Direct GPU Inference ({precision})', fontweight='bold')
for bar, val in zip(bars3, values_direct):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f'{val:.1f}ms', ha='center', fontsize=9, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='y')

# Panel 4: GPU Utilization over time
ax4 = fig.add_subplot(gs[1, 0])
if gpu_samples and len(gpu_samples) > 1:
    t_samples = [s['ts'] - gpu_samples[0]['ts'] for s in gpu_samples]
    util_data = [s['gpu_util_pct'] for s in gpu_samples]
    ax4.fill_between(t_samples, util_data, alpha=0.4, color='#8b5cf6')
    ax4.plot(t_samples, util_data, color='#8b5cf6', linewidth=1.5)
    ax4.set_xlabel('Time (s)')
    ax4.set_ylabel('GPU Utilization (%)')
    ax4.set_title('GPU Utilization During Sustained Load', fontweight='bold')
    ax4.set_ylim(0, 100)
    ax4.grid(True, alpha=0.3)
else:
    ax4.text(0.5, 0.5, 'No GPU utilization data', ha='center', va='center',
             transform=ax4.transAxes, fontsize=12, color='gray')
    ax4.set_title('GPU Utilization', fontweight='bold')

# Panel 5: GPU Memory over time
ax5 = fig.add_subplot(gs[1, 1])
if gpu_samples and len(gpu_samples) > 1:
    t_samples = [s['ts'] - gpu_samples[0]['ts'] for s in gpu_samples]
    mem_data = [s['mem_used_gb'] for s in gpu_samples]
    ax5.fill_between(t_samples, mem_data, alpha=0.4, color='#06b6d4')
    ax5.plot(t_samples, mem_data, color='#06b6d4', linewidth=1.5)
    ax5.axhline(y=gpu_samples[0]['mem_total_gb'], color='red', linestyle='--',
                alpha=0.5, label=f'Total ({gpu_samples[0]["mem_total_gb"]:.1f} GB)')
    ax5.set_xlabel('Time (s)')
    ax5.set_ylabel('Memory Used (GB)')
    ax5.set_title('GPU Memory During Sustained Load', fontweight='bold')
    ax5.legend()
    ax5.grid(True, alpha=0.3)
else:
    ax5.text(0.5, 0.5, 'No GPU memory data', ha='center', va='center',
             transform=ax5.transAxes, fontsize=12, color='gray')
    ax5.set_title('GPU Memory', fontweight='bold')

# Panel 6: Summary table
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('off')

peak_util = max([s['gpu_util_pct'] for s in gpu_samples]) if gpu_samples else 'N/A'
peak_rps = max(rps) if rps else 'N/A'

table_data = [
    ['Metric', 'Value'],
    ['GPU', gpu_label[:22] if gpu_label else 'N/A'],
    ['Precision', precision],
    ['Direct FPS', str(direct_results['fps'])],
    ['P50 latency', f"{direct_results['median_ms']} ms"],
    ['P99 latency', f"{direct_results['p99_ms']} ms"],
    [f'Peak RPS (c={concurrencies[-1]})', str(peak_rps)],
    ['GPU peak util', f"{peak_util}%"],
]
table = ax6.table(cellText=table_data, cellLoc='left', loc='center', bbox=[0, 0, 1, 1])
table.auto_set_font_size(False)
table.set_fontsize(10)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#1e3a5f')
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#f0f4f8')
ax6.set_title('Summary', fontweight='bold', pad=8)

fig.suptitle(
    f'YOLOv8-Large Wafer Defect Detection — GPU Inference Benchmark\n'
    f'{precision} | {gpu_label}',
    fontsize=14, fontweight='bold', y=1.01
)

plt.savefig('gpu_stack_benchmark.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Chart saved: gpu_stack_benchmark.png')


In [ ]:
# ─── Cell 9: Save all results to JSON + CSV ───────────────────────────────────
import json, pandas as pd

# Combined JSON
all_results = {
    'direct_inference': direct_results,
    'load_test': load_test_results,
    'gpu_monitoring': gpu_samples,
}
with open('gpu_stack_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

# Load test summary CSV
df_load = pd.DataFrame(load_test_results)
df_load.to_csv('load_test_summary.csv', index=False)

# GPU monitoring CSV
if gpu_samples:
    df_gpu = pd.DataFrame(gpu_samples)
    df_gpu.to_csv('gpu_monitoring.csv', index=False)

print('=== Results saved ===')
print('  gpu_stack_results.json')
print('  load_test_summary.csv')
print('  gpu_monitoring.csv')
print()
print('=== Load Test Summary ===')
print(df_load[['concurrency','throughput_rps','p50_ms','p95_ms','p99_ms','error_rate_pct']].to_string(index=False))

In [ ]:
# ─── Cell 10: Stop server + Download all results ─────────────────────────────
import shutil, os
from google.colab import files

# Stop server gracefully
if server_proc is not None:
    if server_proc.poll() is None:
        server_proc.terminate()
        try:
            server_proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            server_proc.kill()
            server_proc.wait()
        print('Server stopped.')
    else:
        print(f'Server was already stopped (exit code: {server_proc.returncode})')
else:
    print('No server process to stop.')

# Bundle results
os.makedirs('gpu_stack_results', exist_ok=True)
for fname in ['gpu_stack_results.json', 'load_test_summary.csv', 'gpu_monitoring.csv', 'gpu_stack_benchmark.png']:
    if os.path.exists(fname):
        shutil.copy(fname, 'gpu_stack_results/')
        print(f'  Copied: {fname}')

shutil.make_archive('gpu_stack_results', 'zip', 'gpu_stack_results')
print()
print('=== DOWNLOAD STARTED ===')
files.download('gpu_stack_results.zip')
print()
print('Share the printed summary table (Cell 9) with Copilot to update README.')


## After Running — Bring Back to Your Local Repo

Once you've downloaded `gpu_stack_results.zip`, share with Copilot:
- The printed **Load Test Summary** table (text output from Cell 9)
- The **direct inference numbers** (Cell 3 output)
- The **GPU info** (which GPU, memory)

Copilot will then:
- Update `README.md` with real benchmark numbers
- Copy `gpu_stack_benchmark.png` to `outputs/`
- Final commit — project is 100% complete with real GPU proof